# Skeletonized Morphology Visualization Notebook

Copyright (c) 2025 Open Brain Institute

+ Author(s): 
    - Michael W. Reimann < michael.reimann@openbraininstitute.org >
    - Marwan Abdellah < marwan.abdellah@openbraininstitute.org >

Last modified: 11.2025

## Imports and setting up platform authentication

We begin by importing required packages.

In [ ]:
import os
import obi_auth
import obi_notebook.get_entities
from obi_notebook.get_projects import get_projects

from entitysdk import Client
from entitysdk.models import CellMorphology, Subject, EMCellMesh

from morph_spines import load_morphology_with_spines
import morph_spines_visualizer

import pandas as pd
from IPython.display import display

## Authentication and project selection
We authenticate with the OBI platform.
### Project selection
Select from the dropdown menu the project that the output (a morphology with extracted spines) should be registered to. It will be available only in that project context. 

The widget lists all projects you have access to.

In [ ]:
env_string = "staging"
token = obi_auth.get_token(environment=env_string)
project_context = get_projects(token=token, env=env_string)

## Set up clients

With the information provided above we assemble a client that can interact with the entity database.

In [ ]:
client = Client(environment=env_string, token_manager=token, project_context=project_context)

## Display skeletonized morphologies table

In [ ]:
microns_subject = client.search_entity(entity_type=Subject, query={"name": "IARPA MICrONS mouse"}).one()
cell_ids = []
cell_ids = obi_notebook.get_entities.get_entities("cell-morphology", token=token, result=cell_ids,
                                                  env=env_string, project_context=project_context, page_size=50)

## Morphology selection

In [ ]:
root = "download"
os.makedirs(root, exist_ok=True)

cell_entity = client.get_entity(entity_id=cell_ids[0], entity_type=CellMorphology, 
                                project_context=project_context)
assets = [asset for asset in cell_entity.assets if asset.label == "morphology_with_spines"]
if len(assets) == 0:
    raise(RuntimeError("The selected morphology does not have spines. Please select a 'morphology-with-spines!'"))

path_dl_morph = client.download_file(entity_id=cell_entity.id, entity_type=CellMorphology, asset_id=assets[0].id, output_path=root)


# TODO: This has to be replaced to avoid loading the morphology twice 
m = load_morphology_with_spines(path_dl_morph)
morphology = m
# This will be improved once we have proper provenance!
try:
    pt_root_id = int(m.morphology.name.split("-")[-1])
except:
    int(cell_entity.description.split()[-1][:-1])

microns_mesh = list(client.search_entity(entity_type=EMCellMesh, query={
    "dense_reconstruction_cell_id": pt_root_id
}))[0]


path_dl_mesh = client.download_file(entity_id=microns_mesh.id,
                                    entity_type=EMCellMesh,
                                    asset_id=microns_mesh.assets[0].id,
                                    output_path=root)

## Visualize the spiny morphology 
This visualization plots the skeleton of the resulting morphology combined with the EM mesh. Users can then select any section with spines, and the spine meshes will pop-up in the scene. 

In [ ]:
# Visualize the data 
morph_spines_visualizer.visualize_morphology_with_point_cloud(
    morphology_path=path_dl_morph, 
    mesh_path=path_dl_mesh
)